# Taxi Demand Forecasting Using Spatio-Temporal Behavioural Modelling

## 1. Problem Formulation

Taxi systems generate large volumes of trip data across time and space. Demand varies significantly by location and time, making accurate forecasting essential for efficient fleet management and urban planning.

Taxi demand shows strong temporal dependence, where past demand influences future demand, along with seasonal and spatial effects.

### Analytical Problem
How can historical taxi trip data be used to model spatio-temporal demand patterns and predict future taxi demand across different locations and time periods?

### Objectives
- Model taxi demand across space and time  
- Capture temporal dependencies using lag and rolling features  
- Learn relationships between demand and contextual variables  
- Forecast future demand for operational planning  


## 2. Architecture Design

- **Data Source:** NYC Yellow Taxi Trip Dataset (2023)  
- **Processing Framework:** Apache Spark  

### Justification
Spark enables scalable processing of large spatio-temporal datasets and supports end-to-end machine learning pipelines.

## 3. Data Processing Strategy

A batch processing approach is used to convert raw trip data into aggregated demand per location and time interval.

### Output
- Spatio-temporal demand dataset  
- Forecasted demand values  


## 4. End-to-End Machine Learning Pipeline

1. Data Loading and inspection  
2. Data cleaning (removal of invalid trips)  
3. Feature engineering (time, lag, seasonal features)  
4. Demand aggregation (location-time grouping)  
5. Exploratory data analysis  
6. Model training (GBT regression)  
7. Model evaluation (RMSE, MAE, R²)  
8. Demand prediction  
9. Residual and error analysis  
10. Feature importance analysis  


## Importing Libriaries

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from math import pi

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os

## Loading data

In [3]:
spark = SparkSession.builder.appName("DemandForecastingFull").getOrCreate()

# Load dataset
df = spark.read.parquet("yellow_tripdata_2023-01.parquet")

# View sample data
print("RAW DATA SAMPLE")
df.show(5)

# Check schema
print("SCHEMA")
df.printSchema()

# Count rows
print("ROW COUNT:", df.count())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 20:23:21 WARN Utils: Your hostname, DESKTOP-7C60LLH, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/22 20:23:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 20:23:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

RAW DATA SAMPLE
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2023-01-01 00:32:10|  2023-01-01 00:40:36|            1.0|         0.97|       1.0|                 N|         161|         141|           2|        9.3|  1.0| 

In [4]:
from pyspark.sql.functions import col, count, mean, stddev, min, max, skewness, kurtosis
import pyspark.sql.functions as F

# Basic dataset structure
print("Number of Rows:", df.count())
print("Number of Columns:", len(df.columns))
df.printSchema()

# Sample records
df.show(5, truncate=False)

# Summary statistics for numeric columns
numeric_cols = [
    "passenger_count", "trip_distance", "fare_amount",
    "extra", "mta_tax", "tip_amount",
    "tolls_amount", "improvement_surcharge",
    "total_amount", "congestion_surcharge", "airport_fee"
]

df.select(numeric_cols).describe().show()

# Advanced statistics (distribution shape)
df.select([
    mean("trip_distance").alias("mean_trip_distance"),
    stddev("trip_distance").alias("std_trip_distance"),
    min("trip_distance").alias("min_trip_distance"),
    max("trip_distance").alias("max_trip_distance"),
    skewness("trip_distance").alias("skew_trip_distance"),
    kurtosis("trip_distance").alias("kurt_trip_distance"),

    mean("fare_amount").alias("mean_fare"),
    stddev("fare_amount").alias("std_fare"),
    min("fare_amount").alias("min_fare"),
    max("fare_amount").alias("max_fare"),
    skewness("fare_amount").alias("skew_fare"),
    kurtosis("fare_amount").alias("kurt_fare"),
]).show(truncate=False)

# Missing values per column
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show(truncate=False)

# Duplicate check
total_count = df.count()
distinct_count = df.distinct().count()

print("Total Records:", total_count)
print("Distinct Records:", distinct_count)
print("Duplicate Records:", total_count - distinct_count)

# Outlier detection (trip distance using IQR)
q1, q3 = df.approxQuantile("trip_distance", [0.25, 0.75], 0.01)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = df.filter(
    (col("trip_distance") < lower_bound) |
    (col("trip_distance") > upper_bound)
)

print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("Outlier Count:", outliers.count())

# Dataset time range
df.select(
    F.min("tpep_pickup_datetime").alias("start_time"),
    F.max("tpep_pickup_datetime").alias("end_time")
).show(truncate=False)

Number of Rows: 3066766
Number of Columns: 19
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+-

26/04/22 20:23:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+
|summary|   passenger_count|     trip_distance|       fare_amount|             extra|            mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        airport_fee|
+-------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+
|  count|           2995023|           3066766|           3066766|           3066766|            3066766|           3066766|           3066766|              3066766|           3066766|             2995023|            2995023|
|   mean|1.3625321074328978|3.8473420306601414| 18.36706861234247|1.5378415633928424|0.488289977

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|0       |0                   |0                    |71743          |0            |71743     |71743             |0           |0           |0           |0          |0    |0      |0        

Total Records: 3066766
Distinct Records: 3066766
Duplicate Records: 0


Lower Bound: -2.1999999999999997
Upper Bound: 6.6
Outlier Count: 396526
+-------------------+-------------------+
|start_time         |end_time           |
+-------------------+-------------------+
|2008-12-31 23:01:42|2023-02-01 00:56:53|
+-------------------+-------------------+



## Data Cleaning

In [5]:
# Summary before cleaning
print("=== BEFORE CLEANING ===")
df.select(
    mean("trip_distance").alias("mean_distance"),
    min("trip_distance").alias("min_distance"),
    max("trip_distance").alias("max_distance")
).show()

# Apply data cleaning filters based on domain constraints
df_clean = df.filter(
    (col("trip_distance") > 0) &
    (col("trip_distance") < 50) &
    (col("fare_amount") > 0) &
    (col("fare_amount") < 200) &
    (col("passenger_count").between(1, 6)) &
    (col("total_amount") > 0)
)

# Summary after cleaning
print("=== AFTER CLEANING ===")
df_clean.select(
    mean("trip_distance").alias("mean_distance"),
    min("trip_distance").alias("min_distance"),
    max("trip_distance").alias("max_distance")
).show()

# Number of removed records
print("Rows removed:", df.count() - df_clean.count())

=== BEFORE CLEANING ===
+------------------+------------+------------+
|     mean_distance|min_distance|max_distance|
+------------------+------------+------------+
|3.8473420306601414|         0.0|   258928.15|
+------------------+------------+------------+

=== AFTER CLEANING ===
+------------------+------------+------------+
|     mean_distance|min_distance|max_distance|
+------------------+------------+------------+
|3.4114124749341053|        0.01|       49.98|
+------------------+------------+------------+

Rows removed: 183352


## Feature Engineering

In [6]:
# Extract temporal features from pickup timestamp to capture time-based demand patterns
df_feat = df_clean.withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
    .withColumn("day_of_week", dayofweek("tpep_pickup_datetime")) \
    .withColumn("day_of_month", dayofmonth("tpep_pickup_datetime")) \
    .withColumn("month", month("tpep_pickup_datetime")) \
    .withColumn("year", year("tpep_pickup_datetime")) \

# Apply cyclical encoding to preserve periodic nature of time variables (hour and day)
# This helps the model understand that time wraps around (e.g., 23:00 → 00:00)
    .withColumn("hour_sin", sin(2 * pi * col("pickup_hour") / 24)) \
    .withColumn("hour_cos", cos(2 * pi * col("pickup_hour") / 24)) \
    .withColumn("dow_sin", sin(2 * pi * col("day_of_week") / 7)) \
    .withColumn("dow_cos", cos(2 * pi * col("day_of_week") / 7))

IndentationError: unexpected indent (2884996260.py, line 10)

## Demand Aggregation

In [ ]:
# Aggregate trip records into demand counts per location and time unit
# Each row now represents demand at a specific location-hour combination
demand_df = df_feat.groupBy(
    "PULocationID",
    "pickup_hour",
    "day_of_week",
    "day_of_month",
    "month",
    "year",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos"
).agg(count("*").alias("demand"))

# Display sample aggregated demand data for verification
print("DEMAND SAMPLE")
demand_df.show(5)

## More FEATURE ENGINEERING

In [ ]:
# Create behavioural features to capture real-world mobility patterns
demand_df = demand_df.withColumn(
    "is_weekend",
    when(col("day_of_week").isin([1, 7]), 1).otherwise(0)
).withColumn(
    "is_peak_hour",
    when(
        (col("pickup_hour").between(7, 9)) |
        (col("pickup_hour").between(17, 19)),
        1
    ).otherwise(0)
)

# Apply log transformation to stabilize highly skewed demand distribution
demand_df = demand_df.withColumn("label", log1p(col("demand")))

# Generate summary statistics for exploratory analysis
print("EDA SUMMARY")
demand_df.describe().show()

## Correlation heatmap

In [ ]:
# Create directory to store visual outputs if it does not exist
plot_dir = "plots"
os.makedirs(plot_dir, exist_ok=True)

# Sample a subset of data for correlation analysis (to reduce computation cost)
corr_pd = demand_df.select(
    "demand",
    "pickup_hour",
    "day_of_week",
    "day_of_month",
    "month",
    "is_weekend",
    "is_peak_hour"
).sample(0.05).toPandas()

# Compute and visualize correlation structure between variables
plt.figure(figsize=(8,5))
sns.heatmap(corr_pd.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")

# Save plot for inclusion in report
plt.savefig(f"{plot_dir}/correlation.png")
plt.show()

In [ ]:
# Apply log transformation to the demand variable
# This reduces skewness and stabilizes variance in the target distribution
demand_df = demand_df.withColumn("label", log1p(col("demand")))

In [ ]:
# Sample demand values for distribution analysis
dist_pd = demand_df.select("demand").sample(0.05).toPandas()

# Apply log transformation to reduce skewness and improve visualization clarity
dist_pd["log_demand"] = np.log1p(dist_pd["demand"])

In [ ]:
# Visualize distribution of raw vs log-transformed demand
plt.figure(figsize=(10,4))

# Raw demand distribution (highly skewed)
plt.subplot(1,2,1)
sns.histplot(dist_pd["demand"], bins=50)
plt.title("Raw Demand")

# Log-transformed demand distribution (reduced skewness)
plt.subplot(1,2,2)
sns.histplot(dist_pd["log_demand"], bins=50)
plt.title("Log Demand")

# Save figure for reporting
plt.savefig(f"{plot_dir}/demand_distribution.png")
plt.show()

## Feature Selection

In [ ]:
# Define final feature set used for model training
# These include spatial, cyclical temporal, and behavioral features

feature_cols = [
    "PULocationID",

    # Cyclical time representation (captures periodic patterns in demand)
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",

    # Calendar feature (captures intra-month variation)
    "day_of_month",

    # Behavioral indicators (capture human mobility patterns)
    "is_weekend",
    "is_peak_hour"
]

## Train/Test split 

In [ ]:
# Split dataset into training and testing sets using a time-based approach
# This simulates real-world forecasting where future demand is unknown

train = demand_df.filter(col("day_of_month") <= 25)   # historical period for learning
test = demand_df.filter(col("day_of_month") > 25)     # future period for evaluation

# Check dataset sizes after split
print("TRAIN:", train.count())   # number of training records
print("TEST:", test.count())     # number of testing records

## Features Assembler

In [ ]:
# Define final feature set used for model training
# Combines spatial, temporal, cyclical, and behavioral features

feature_cols = [
    "PULocationID",
    "pickup_hour",
    "day_of_week",
    "day_of_month",
    "month",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "is_weekend",
    "is_peak_hour"
]

# Convert multiple feature columns into a single feature vector for ML models
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

## Model

In [ ]:
# Import Gradient Boosted Tree regressor for non-linear demand modeling
from pyspark.ml.regression import GBTRegressor

# Define GBT model with tuned hyperparameters for complexity and generalization control
gbt = GBTRegressor(
    labelCol="label",        # target variable (log-transformed demand)
    featuresCol="features",  # input feature vector
    maxIter=60,             # number of boosting iterations (trees)
    maxDepth=7,             # depth of each tree (controls model complexity)
    maxBins=256             # handles high-cardinality features like location IDs
)

In [ ]:
# Import Pipeline to chain feature engineering and model training steps
from pyspark.ml import Pipeline

# Create a machine learning pipeline
# Step 1: assemble features into a single vector
# Step 2: train Gradient Boosted Tree model
pipeline = Pipeline(stages=[assembler, gbt])

In [ ]:
# Train the full pipeline on the training dataset
# This applies feature assembly + fits the GBT model

model = pipeline.fit(train)

# Confirmation message after successful training
print("MODEL TRAINED")

## Predictions evaluation

In [ ]:
# Compute shared axis limits for evaluation plots
# Ensures fair visual comparison between actual and predicted values

import numpy as np

min_val = np.minimum(
    pred_pd["label"].min(),
    pred_pd["prediction"].min()
)

max_val = np.maximum(
    pred_pd["label"].max(),
    pred_pd["prediction"].max()
)

In [ ]:
# Visual comparison of actual vs predicted demand values

plt.figure(figsize=(5,5))

# Scatter plot of predictions against actual values
plt.scatter(pred_pd["label"], pred_pd["prediction"], alpha=0.3)

# Reference line for perfect predictions (y = x)
plt.plot([min_val, max_val], [min_val, max_val], color="red")

# Axis labels and title
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted Demand")

plt.show()

## Evaluation

In [ ]:
# Generate predictions on the test dataset using the trained model
pred = model.transform(test)

# View sample predictions for quick inspection
print("PREDICTIONS SAMPLE")
pred.select("label", "prediction").show(10)

# Import evaluator for regression metrics
evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction")

# Compute evaluation metrics
print("RMSE:", evaluator.setMetricName("rmse").evaluate(pred))
print("MAE:", evaluator.setMetricName("mae").evaluate(pred))
print("R2:", evaluator.setMetricName("r2").evaluate(pred))

# Convert sample predictions to Pandas for visualization
pred_pd = pred.select("label", "prediction").sample(0.1).toPandas()

In [ ]:
# Plot relationship between actual and predicted demand values

plt.figure(figsize=(6,5))

# Scatter plot of predictions vs actual values
plt.scatter(pred_pd["label"], pred_pd["prediction"])

# Plot title and axis labels
plt.title("Actual vs Predicted")
plt.xlabel("Actual")
plt.ylabel("Predicted")

# Save figure for report documentation
plt.savefig(f"{plot_dir}/actual_vs_predicted.png")

plt.show()

In [ ]:
# Compute prediction errors (residuals)
pred_pd["error"] = pred_pd["label"] - pred_pd["prediction"]

# Visualize distribution of residuals to assess model bias and error spread
plt.figure(figsize=(6,4))
sns.histplot(pred_pd["error"], bins=50)

# Plot title
plt.title("Residuals")

# Save figure for reporting
plt.savefig(f"{plot_dir}/residuals.png")

plt.show()

## Feature importance

In [ ]:
# Extract trained GBT model from pipeline
gbt_model = model.stages[-1]

# Get feature importance scores from the trained model
importances = gbt_model.featureImportances.toArray()

# Map feature names to their importance values
features = assembler.getInputCols()

# Create a structured table of feature importance rankings
fi = pd.DataFrame({
    "feature": features,
    "importance": importances
}).sort_values("importance", ascending=False)

In [ ]:
# Visualize feature importance from the trained GBT model

plt.figure(figsize=(8,5))

# Bar plot showing contribution of each feature to model predictions
sns.barplot(x="importance", y="feature", data=fi)

# Plot title
plt.title("Feature Importance")

# Save figure for report inclusion
plt.savefig(f"{plot_dir}/feature_importance.png")

plt.show()

In [ ]:
# Extract feature importance from trained GBT model
importances = model.stages[-1].featureImportances

# Map importance values to feature names
fi = pd.DataFrame({
    "feature": feature_cols,
    "importance": importances.toArray()
}).sort_values("importance", ascending=False)

# Display feature importance ranking
print("FEATURE IMPORTANCE")
print(fi)